# Lab 09 solution, notebook 2: a typed answer that can say "not found", and a refund edited before it is filed

The worked answers to the two challenge tasks of `09b_langchain_at_the_desk.ipynb`. The setup repeats the
notebook's model, lookups and helper, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import sqlite3
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(model=MODEL, reasoning_effort=EFFORT, timeout=60)

SYSTEM = ("You help the support reps of a music shop. Look up every fact with the tools; never answer from memory. "
          "Answer in one or two sentences.")


def describe(message):
    """One line that says what a message holds."""
    if message.type == "ai" and message.tool_calls:
        return "asks for " + str([(call["name"], call["args"]) for call in message.tool_calls])
    return message.text

In [3]:
DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name: str) -> list:
    """Find a shop customer by full name. Returns a list of matches with their customer_id."""
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

## Task 1: A typed answer that can say "not found"

In the notebook the agent filled a complete `DeskAnswer` for *Helena Holy*, a customer the shop does not have. Two
things were missing. The lookup returned an empty list, which reached the model as an empty result, so the model
was never told "no match". And the class had no field in which to say "not found".

First the lookup. `find_customer_clearly` calls `find_customer`. `or` returns the list when it has something in
it, and the sentence when the list is empty.

In [4]:
def find_customer_clearly(name: str) -> list | str:
    """Find a shop customer by full name. Returns a list of matches with their customer_id."""
    return find_customer(name) or f"No customer is called {name}."


print(find_customer_clearly("Helena Holý"))
print(find_customer_clearly("Helena Holy"))

[{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]
No customer is called Helena Holy.


Then the class: `DeskAnswer` with one more field, `found`, and a description that says when it is true.

In [5]:
from pydantic import BaseModel, Field


class CheckedAnswer(BaseModel):
    """The facts a support rep asked for about one customer."""
    found: bool = Field(description="True only when the customer lookup returned a match")
    customer_name: str = Field(description="The customer's full name, as the database spells it")
    invoice_count: int
    total_usd: float = Field(description="What the customer's invoices add up to, in US dollars")
    support_rep: str

In [6]:
checked_desk = create_agent(
    llm,
    [find_customer_clearly, get_invoices, get_support_rep],
    system_prompt=SYSTEM,
    response_format=CheckedAnswer,
)

QUESTION = "How many invoices does {name} have, what do they add up to, and who is her support rep?"

for name in ["Helena Holy", "Helena Holý"]:
    run = checked_desk.invoke({"messages": [{"role": "user", "content": QUESTION.format(name=name)}]})
    print(repr(run["structured_response"]))
    for message in run["messages"][1:-1]:
        print(f"   {message.type:5} | {describe(message)}")
    print()

CheckedAnswer(found=False, customer_name='', invoice_count=0, total_usd=0.0, support_rep='')
   ai    | asks for [('find_customer_clearly', {'name': 'Helena Holy'})]
   tool  | No customer is called Helena Holy.

CheckedAnswer(found=True, customer_name='Helena Holý', invoice_count=7, total_usd=49.62, support_rep='Steve Johnson')
   ai    | asks for [('find_customer_clearly', {'name': 'Helena Holý'})]
   tool  | [{"customer_id": 6, "name": "Helena Holý", "country": "Czech Republic"}]
   ai    | asks for [('get_invoices', {'customer_id': 6}), ('get_support_rep', {'customer_id': 6})]
   tool  | {"customer_id": 6, "invoice_count": 7, "total": 49.62}
   tool  | {"customer_id": 6, "support_rep": "Steve Johnson"}



Under each record the cell prints the run's messages. It leaves out the first, which is the question, and the last,
which is the record again as JSON text.

For *Helena Holy* the lookup's result is now a sentence, *No customer is called Helena Holy.*, and the agent
stopped there: one tool call, `found=False`, and empty or zero values in the other fields. For *Helena Holý* the
run is the one from Part A, and the record has `found=True`, 7 invoices, 49.62 and Steve Johnson. On this model
the *Helena Holy* result was the same in every run we made (eight).

**Key lesson:** the lookup now says what happened in words, and the type has a place for the answer "not found".
Our code reads one field, `found`, before it uses the others. The numbers in a record with `found=False` are
placeholders the type forced the model to write.

## Task 2: Edit a refund before it is filed

The write tool, the reviewer's lookup and the careful agent, as in Part D.

In [7]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

REFUND_REQUESTS = []


def file_refund_request(invoice_id: int, amount: float, reason: str) -> str:
    """File a refund request for one invoice, in US dollars. A person reviews it before any money moves."""
    REFUND_REQUESTS.append({"invoice_id": invoice_id, "amount": amount, "reason": reason})
    return f"refund request {len(REFUND_REQUESTS)} filed: invoice {invoice_id}, {amount} US dollars"


def show_invoice(invoice_id):
    """Print one invoice from the shop's database: its number, customer, date and total."""
    row = conn.execute(
        "SELECT i.InvoiceId, c.FirstName || ' ' || c.LastName, date(i.InvoiceDate), i.Total "
        "FROM invoices i JOIN customers c ON c.CustomerId = i.CustomerId WHERE i.InvoiceId = ?",
        (invoice_id,)).fetchone()
    print("the shop's record:", row)


approval = HumanInTheLoopMiddleware(
    interrupt_on={"file_refund_request": {"allowed_decisions": ["approve", "edit", "reject"]}},
    interrupt_mode="per_call",
)

careful_desk = create_agent(
    llm,
    [find_customer, get_invoices, get_support_rep, file_refund_request],
    system_prompt=SYSTEM,
    middleware=[approval],
    checkpointer=InMemorySaver(),
)

In [8]:
REQUEST = "Puja Srivastava was charged twice for invoice 229. Please file a refund request for 18.36 US dollars."
config = {"configurable": {"thread_id": "refund-puja-229"}}

paused = careful_desk.invoke({"messages": [{"role": "user", "content": REQUEST}]}, config, version="v2")

waiting = paused.interrupts[0]
print("waiting:", waiting.value["name"], waiting.value["args"])
show_invoice(229)

waiting: file_refund_request {'invoice_id': 229, 'amount': 18.36, 'reason': 'Duplicate charge reported by Puja Srivastava.'}
the shop's record: (229, 'Puja Srivastava', '2011-09-30', 13.86)


The waiting call asks for 18.36, the amount in the rep's message. The shop's record says invoice 229 is Puja
Srivastava's and its total is 13.86: the right customer and the right invoice, with two digits swapped in the
amount.

An `edit` decision carries the call that should run in place of the model's: the tool's `name` and its full
`args`. We copy the model's arguments and change one value, so the invoice and the reason stay as they were.
`{**a, "amount": 13.86}` builds a new dictionary with everything in `a`, and `amount` replaced.

In [9]:
corrected = {**waiting.value["args"], "amount": 13.86}

decision = {"type": "edit", "edited_action": {"name": "file_refund_request", "args": corrected}}

edited = careful_desk.invoke(Command(resume={waiting.id: decision}), config, version="v2")

print("tool message:", edited.value["messages"][-2].text)
print()
print("reply:", edited.value["messages"][-1].text)
print()
print("refund requests:", REFUND_REQUESTS)

tool message: Note: a human reviewer replaced this tool call before it ran. The call recorded in your message is the one you produced, not the one that executed. This was intentional and authorized. Do not re-issue your original call. Executed instead: file_refund_request with arguments {"invoice_id": 229, "amount": 13.86, "reason": "Duplicate charge reported by Puja Srivastava."}.

Tool response:
refund request 1 filed: invoice 229, 13.86 US dollars

reply: A refund request was filed for invoice 229, but the amount submitted was **$13.86**, not the requested $18.36. A human reviewer changed the request before it was submitted.

refund requests: [{'invoice_id': 229, 'amount': 13.86, 'reason': 'Duplicate charge reported by Puja Srivastava.'}]


The list holds one request, for invoice 229 and 13.86: the tool ran with our corrected arguments, not with the
model's.

The tool message shows what the model was told. LangChain put a note in front of the tool's result: *Note: a
human reviewer replaced this tool call before it ran.* The note ends with the call that ran, with the amount
13.86, and the tool's own result follows under *Tool response:*. That is how the model can tell the rep what
happened: *A refund request was filed for invoice 229, but the amount submitted was \$13.86, not the requested
\$18.36. A human reviewer changed the request before it was submitted.*

**Key lesson:** a reviewer has three decisions, and `edit` is the one for a request that is right apart from one
value. The model proposed what the rep typed; the person corrected it against the record; the tool ran once, with
the corrected call.